In [58]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier

import sys
sys.path.append("..")
from src.transforms import get_preprocessor_fixed, categorical_features

SEED = 42

In [59]:
train_df = pd.read_csv("../artifacts/data/train.csv")
hold_df  = pd.read_csv("../artifacts/data/holdout.csv")

X_trainval = train_df.drop(columns=["num"])
y_trainval = train_df["num"].values

X_holdout  = hold_df.drop(columns=["num"])
y_holdout  = hold_df["num"].values

print("Trainval size:", X_trainval.shape)
print("Holdout size:", X_holdout.shape)

Trainval size: (736, 15)
Holdout size: (184, 15)


In [60]:
mean_imp = pd.read_csv("../artifacts/feature_importance_mean.csv")

top_features = mean_imp.sort_values("rank")['feature'].tolist()[:15]
print("Top-15 features:", top_features)

Top-15 features: ['cp_asymptomatic', 'exang_False', 'cp_atypical angina', 'chol', 'sex_Female', 'oldpeak', 'thal_normal', 'age', 'ca', 'slope_upsloping', 'thalch', 'fbs_False', 'restecg_normal', 'restecg_st-t abnormality', 'thal_reversable defect']


In [61]:
categories_list = [
    sorted(X_trainval[c].dropna().astype(str).unique().tolist())
    for c in categorical_features
]

preprocessor_fixed = get_preprocessor_fixed(categories_list)

In [62]:
def get_feature_names(ct):
    names=[]
    for name, transformer, cols in ct.transformers_:
        if hasattr(transformer,'named_steps'):
            last = list(transformer.named_steps.values())[-1]
            if hasattr(last,'get_feature_names_out'):
                names.extend(last.get_feature_names_out(cols))
            else:
                names.extend(cols)
        else:
            if hasattr(transformer,'get_feature_names_out'):
                names.extend(transformer.get_feature_names_out(cols))
            else:
                names.extend(cols)
    return names

preprocessor_fixed.fit(X_trainval)
full_feature_names = get_feature_names(preprocessor_fixed)
print("Transformed feature count:", len(full_feature_names))

Transformed feature count: 27


In [63]:
top_indices = [full_feature_names.index(f) for f in top_features]
print("Top indices:", top_indices)

Top indices: [8, 17, 9, 2, 6, 4, 23, 0, 5, 21, 3, 12, 15, 16, 24]


In [64]:
from sklearn.base import BaseEstimator, TransformerMixin

class ColumnSelector(BaseEstimator, TransformerMixin):
    def __init__(self, indices):
        self.indices = indices
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return X[:, self.indices]


In [65]:
import json

best = json.load(open("../artifacts/tuning_optuna/optuna_k15_best.json"))
final_params = best["best_params"]

In [66]:
final_pipe = Pipeline([
    ("preprocessor", preprocessor_fixed),
    ("selector", ColumnSelector(top_indices)),
    ("model", XGBClassifier(**final_params))
])

final_pipe.fit(X_trainval, y_trainval)

print("Final lightweight model trained.")


Final lightweight model trained.


Calibration complete.


In [67]:
from sklearn.metrics import roc_auc_score, average_precision_score

probs = final_pipe.predict_proba(X_holdout)[:,1]

auc_hold = roc_auc_score(y_holdout, probs)
ap_hold = average_precision_score(y_holdout, probs)

print("Holdout ROC-AUC:", auc_hold)
print("Holdout AUC-PR:", ap_hold)


Holdout ROC-AUC: 0.9127211860353898
Holdout AUC-PR: 0.9181048346938625


In [68]:
joblib.dump(final_pipe, "../artifacts/models/xgb_k15_final.joblib")
print("Saved final k=15 model.")

Saved final k=15 model.
